In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

DATA_PATH = "spam.csv"

df = pd.read_csv(DATA_PATH, encoding="latin-1")

# Support common dataset column names.
label_col = next(
    (c for c in df.columns if c.lower() in {"label", "category", "class", "v1", "spam"}),
    None
)
text_col = next(
    (c for c in df.columns if c.lower() in {"text", "message", "email", "v2"}),
    None
)

if label_col is None or text_col is None:
    raise ValueError(
        f"Could not identify label/text columns. Available columns: {list(df.columns)}. "
        "Set label_col and text_col manually."
    )

data = df[[label_col, text_col]].dropna().copy()
data.columns = ["label", "text"]
data["label"] = data["label"].astype(str).str.strip().str.lower()
data["text"] = data["text"].astype(str)

# Normalize common label variants while leaving other binary labels unchanged.
data["label"] = data["label"].replace({
    "ham": "not spam",
    "not_spam": "not spam",
    "non-spam": "not spam",
    "nonspam": "not spam"
})

X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

models = {
    "K-Nearest Neighbors": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=10000)),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5, weights="distance", metric="cosine", algorithm="brute"
        ))
    ]),
    "Support Vector Machine": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=10000)),
        ("classifier", LinearSVC(class_weight="balanced", random_state=42))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (spam)": precision_score(
            y_test, predictions, pos_label="spam", zero_division=0
        ),
        "Recall (spam)": recall_score(
            y_test, predictions, pos_label="spam", zero_division=0
        ),
        "F1 (spam)": f1_score(
            y_test, predictions, pos_label="spam", zero_division=0
        )
    })

    print(f"\n{name}")
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix (rows = actual, columns = predicted):")
    print(confusion_matrix(y_test, predictions, labels=["not spam", "spam"]))

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(3))

best_model = results_df["F1 (spam)"].idxmax()
print(f"Best model by spam F1-score: {best_model}")
print("Compare spam recall and precision alongside accuracy: "
      "recall measures how many spam emails were caught, while precision "
      "measures how often flagged emails were actually spam.")


K-Nearest Neighbors
              precision    recall  f1-score   support

    not spam       1.00      1.00      1.00         5
        spam       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

Confusion matrix (rows = actual, columns = predicted):
[[5 0]
 [0 5]]

Support Vector Machine
              precision    recall  f1-score   support

    not spam       1.00      1.00      1.00         5
        spam       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

Confusion matrix (rows = actual, columns = predicted):
[[5 0]
 [0 5]]


,Accuracy,Precision (spam),Recall (spam),F1 (spam)
Model,,,,
K-Nearest Neighbors,1.0,1.0,1.0,1.0
Support Vector Machine,1.0,1.0,1.0,1.0


Best model by spam F1-score: K-Nearest Neighbors
Compare spam recall and precision alongside accuracy: recall measures how many spam emails were caught, while precision measures how often flagged emails were actually spam.
